# ArchVis RGB bounded discovery — CPU
Root가 커밋 후 source/input ZIP SHA와 새 snapshot 폴더를 고정한 ready본으로만 실행합니다.
기존 pose notebook/DB/HEAD는 변경하지 않습니다. 실제 추출·학습·전체 CRC 검증은 없습니다.
한 job: body256MiB(초기metadata 포함)/2048HTTP/600초/read512MiB/decode256MiB/RSS1GiB/local+snapshot+회수256MiB,
32member×16batch 이하. 준비 업로드/설치/auth/source 해제 공간과 시간은 phase 밖입니다.
목록 key 증가와 stereo+pose관측 교집합, 실제 추출triplet을 구분합니다. job을 재실행하지 않습니다.
실패/중단 뒤에는 마지막 회수 셀만 실행합니다. 새 job/anchor/예산은 root가 명시 인계합니다.


In [ ]:
# Root가 고정할 값. 빈 pin으로는 실행하지 않습니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
INPUT_ZIP_SHA256 = ""
JOB_ID = "archvis_rgb_discovery_001"
SOURCE_ROOT = "/content/rewind-discovery-source"
INPUT_ROOT = "/content/rewind-discovery-inputs"
WORKSPACE = "/content/rewind-discovery/archvis_rgb_discovery_001"
SNAPSHOT_DIRECTORY = ""  # 새 별도 Drive 폴더; 기존 HEAD/pose 저장소 금지
RESUME = None  # 다음 job만: receipt_path/receipt_file_sha256/anchor를 root가 고정
INPUT_SNAPSHOT_DIRECTORY = None  # 중단 시 이전 정상 anchor 회수에 사용
RECOVERY_ZIP = "/content/archvis_rgb_discovery_001_result.zip"


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert SNAPSHOT_DIRECTORY and not Path(WORKSPACE).exists(), "새 고정 job만 실행"
uploaded = files.upload()  # 고정 source ZIP과 input ZIP 두 개
def selected(pin):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, "고정 ZIP SHA 불일치"
    return rows[0]
def unpack(raw, destination, maximum):
    target = Path(destination)
    assert not target.exists(), "기존 source/input 덮어쓰기 금지"
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum
        names = set()
        for item in rows:
            p = Path(item.filename)
            assert not p.is_absolute() and ".." not in p.parts and "\\" not in item.filename
            assert not stat.S_ISLNK(item.external_attr >> 16) and item.filename not in names
            names.add(item.filename)
        target.mkdir()
        z.extractall(target)
unpack(selected(SOURCE_ZIP_SHA256), SOURCE_ROOT, 64*1024**2)
unpack(selected(INPUT_ZIP_SHA256), INPUT_ROOT, 8*1024**2)
assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
sys.path.insert(0, SOURCE_ROOT)
# 준비 단계이며 실제 이번 개발에서 실행하지 않았습니다.
subprocess.check_call([sys.executable, "-m", "pip", "install", "indexed_gzip==1.10.3", "psutil"])
import indexed_gzip
assert indexed_gzip.__version__ == "1.10.3"
del uploaded


In [ ]:
# 한 번만 실행. auth 뒤에는 600초 phase budget에 준비/scan/snapshot을 포함합니다.
from pathlib import Path
assert not Path(WORKSPACE).exists(), "이미 소비한 job: 회수 셀만 실행"
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
import os
mount = Path("/content/drive")
my_drive = mount/"MyDrive"
assert os.path.ismount(mount) and my_drive.is_dir(), "실제 Drive mount 확인 실패"
snapshot_path = Path(SNAPSHOT_DIRECTORY).resolve()
assert snapshot_path.is_relative_to(my_drive.resolve()) and snapshot_path != my_drive.resolve()
assert not snapshot_path.exists(), "새 별도 snapshot 폴더만 허용"
import google.auth
from google.auth.transport.requests import AuthorizedSession
from urllib3.util.retry import Retry
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
def session_factory():
    session = AuthorizedSession(credentials)
    session._max_refresh_attempts = 0
    for adapter in session.adapters.values():
        adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
    return session
from rewind_discovery.runner import run
config = json.loads((Path(INPUT_ROOT)/"config.json").read_bytes())
report = run(job_id=JOB_ID, selection=config["selection"], expected_metadata=config["expected_metadata"],
    split_paths={s:Path(INPUT_ROOT)/(s+".txt") for s in ("train","validation","test")},
    split_pins=config["split_pins"], pose_catalog=Path(INPUT_ROOT)/"pose_catalog.json",
    pose_sha256=config["pose_catalog_sha256"], workspace=WORKSPACE,
    snapshot_directory=SNAPSHOT_DIRECTORY, session_factory=session_factory, resume=RESUME)
print(json.dumps({k:report.get(k) for k in ("status","body_bytes","requests","after","discovered","last_receipt")}, ensure_ascii=False))


In [ ]:
# 독립 회수: session/auth/scan/run을 다시 호출하지 않습니다.
import hashlib, json, sys, zipfile
from pathlib import Path
from google.colab import files
assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
sys.path.insert(0, SOURCE_ROOT)
from rewind_archive.contracts import digest
from rewind_discovery.recovery import recover
config = json.loads((Path(INPUT_ROOT)/"config.json").read_bytes())
target = Path(RECOVERY_ZIP)
pin_path = target.with_suffix(".sha256")
def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        while data := stream.read(65536): h.update(data)
    return h.hexdigest()
if target.exists():
    assert pin_path.is_file() and file_sha(target) == pin_path.read_text().strip()
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info = json.loads(z.read("recovery.json"))
        assert info["job_id"] == JOB_ID and info["selection_sha256"] == digest(config["selection"])
else:
    assert not pin_path.exists()
    result = recover(WORKSPACE, SNAPSHOT_DIRECTORY, target, job_id=JOB_ID,
        selection_sha256=digest(config["selection"]), input_snapshot_directory=INPUT_SNAPSHOT_DIRECTORY)
    with pin_path.open("x") as stream: stream.write(file_sha(target)+"\n")
print("회수 ZIP SHA256:", file_sha(target))
files.download(str(target))
